# Estimate noise correlations along the x, y, t, T axes before and after pre-denosing

## Define mask

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sys
import os

sys.path.append(os.path.abspath("../src"))
from denoising.eval.empirical_noise_correlations import *
from denoising.figures.empirical_noise_correlations_plots import *
from denoising.data.data_utils import *


# Configuration cell

In [ ]:
# this is the only cell where you need to adapt parameters

subject_ids = ["P03", "P04", "P05", "P06", "P07", "P08"]

#subject_ids = ["Simulated_Lesion_double_1", "Simulated_Lesion_double_2", "Simulated_Lesion_double_3", "Simulated_Lesion_double_4", "Simulated_Lesion_double_5", "Simulated_Lesion_double_6"]

methods = {
    "No denoising": {"type": "raw"},
    "tMPPCA": {"type": "precomputed", "method": "tMPPCA_5D"},
    "Spin SVD": {"type": "callable", "fn": apply_low_rank_5d_to_dataset_list, "kwargs": {"rank": 8}},
}

extra_axes = ["t", "T"]   # oder nur ["t"]

#configure spatial noise mask:
mask_source = "method"  # "raw" or "method": raw from the noisy data, method from the method directly
percentile = 5

## datasets
suffix = "normalized" #"normalized" #"normalized_uncorrelated_noise"
base_dir = "../datasets"

In [ ]:
results = run_noise_analysis_pipeline(
    subject_ids=subject_ids,
    methods=methods,
    extra_axes=extra_axes,
    mask_source=mask_source,
    percentile=percentile,
    suffix=suffix,
    base_dir=base_dir,
)

In [ ]:
plot_spatial_acf_comparison(
    results["spatial_stats_by_method"],
    axes_to_plot=("x", "y", "z"),
    max_lag_plot=15, 
    title="Spatial noise correlations"
)

In [ ]:
plot_acf_comparison(
    results["acf_stats_by_method"],
    axis_idx=results["axis_indices"],
    axis_names=results["extra_axes"],
    title="Noise autocorrelation"
)

# Average absolute FID and repetions in noise voxels compared to all voxels

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

subject_idx = 0
percentile = 10

data = np.load('../datasets/P03/data.npy')  # (x, y, z, t, T)

mask_noise = estimate_noise_masks([data], percentile=percentile, axes=(3, 4))[0]
noise_data = data[mask_noise]  # (N_voxels, t, T)

# --------------------------------------------------
# entlang t
# --------------------------------------------------
mean_noise_t = np.abs(np.mean(noise_data, axis=(0, 2)))   # (t,)
mean_all_t   = np.abs(np.mean(data, axis=(0, 1, 2, 4)))   # (t,)
t = np.arange(mean_noise_t.shape[0])

# --------------------------------------------------
# entlang T
# --------------------------------------------------
mean_noise_T = np.abs(np.mean(noise_data, axis=(0, 1)))   # (T,)
mean_all_T   = np.abs(np.mean(data, axis=(0, 1, 2, 3)))   # (T,)
T = np.arange(mean_noise_T.shape[0])

# --------------------------------------------------
# Plot
# --------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# --- FID (t) ---
axes[0].plot(t, mean_all_t, label="all voxels", marker='o')
axes[0].plot(t, mean_noise_t, label="noise voxels", marker='o')
axes[0].set_xlabel("FID point t")
axes[0].set_ylabel("|mean signal|")
axes[0].set_title("Residual coherent signal along t")
axes[0].grid(True)
axes[0].legend()

# --- Repetitions (T) ---
axes[1].plot(T, mean_all_T, label="all voxels", marker='o')
axes[1].plot(T, mean_noise_T, label="noise voxels", marker='o')
axes[1].set_xlabel("Repetition T")
axes[1].set_ylabel("|mean signal|")
axes[1].set_title("Residual coherent signal along T")
axes[1].grid(True)
axes[1].legend()

plt.suptitle(f"Residual coherent signal (subject {subject_idx})")
plt.tight_layout()
plt.show()